# Scintillator Propagation Speed ($c_{\mathrm{scint}}$) Multi-Scale Variance Analysis Across Operational Eras

This notebook investigates the intrinsic variance, readout asymmetries, and repeatability of $c_{\mathrm{scint}}$ across multiple scales in the SND@LHC detector, **stratified by operational year / category**:
- **Cat II (2022):** 33 reference runs (5119–5408)
- **Cat III (2023):** 10 runs (6080–6668)
- **Cat III (2024):** 10 runs (8638–9880)
- **Cat III (2025):** 10 runs (11181–12224)
- **Cat III (2026):** 10 runs (13085–13520)

### Key Exploratory Topics:
1. **Quality Convergence Filtering:** Rejects unconstrained fits, profile artifacts, and divergent slopes using subsystem-specific goodness-of-fit requirements.
2. **Intra-Bar SiPM Variance:** Variance of extracted $c_{\mathrm{scint}}$ across all SiPMs attached to the *exact same physical bar* (superimposing bar index as a colormap).
3. **Upstream (US) Readout Variance by Year:** Comparative variance of Left SiPMs (0–7), Right SiPMs (8–15), and Combined (Left + Right) for each year.
4. **Veto Readout Variance by Year:** Comparative variance of Left SiPMs, Right SiPMs, and Combined across Veto planes for each year.
5. **Downstream (DS) Readout Variance by Year:** Comparative variance of Left SiPMs (Ch 0), Right SiPMs (Ch 1), and Combined for each year.
6. **Master Cross-Category Synthesis:** Comprehensive summary table and repeatability metrics across all 5 operational eras.


In [ ]:
# Fix JupyROOT post-execute hook crash if ROOT is present
try:
    import ROOT
    try:
        import ROOT._jupyroot.helpers.utils as _jutils
        _jutils.GetRCanvasDrawers = lambda: []
    except (ImportError, AttributeError):
        pass
except ImportError:
    pass

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip and hasattr(_ip, "events"):
        for _cb in list(_ip.events.callbacks.get("post_execute", [])):
            if "CaptureDrawnPrimitives" in str(_cb) or "NotebookDraw" in str(_cb):
                _ip.events.unregister("post_execute", _cb)
except Exception:
    pass

import os
import glob
import json
import concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy import stats

try:
    import ddfUtils.mpl_styling as dms
except ImportError:
    pass

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9.5,
    "figure.titlesize": 14,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

periods_ordered = [
    "Cat II (2022)",
    "Cat III (2023)",
    "Cat III (2024)",
    "Cat III (2025)",
    "Cat III (2026)",
]

period_labels_short = {
    "Cat II (2022)":  "2022",
    "Cat III (2023)": "2023",
    "Cat III (2024)": "2024",
    "Cat III (2025)": "2025",
    "Cat III (2026)": "2026",
}

period_colors = {
    "Cat II (2022)":  "#1f77b4",
    "Cat III (2023)": "#ff7f0e",
    "Cat III (2024)": "#2ca02c",
    "Cat III (2025)": "#d62728",
    "Cat III (2026)": "#9467bd",
}

period_spans = [
    ("Cat II (2022)",  5100,  5430,  "#1f77b4"),
    ("Cat III (2023)", 6050,  6700,  "#ff7f0e"),
    ("Cat III (2024)", 8600,  9900,  "#2ca02c"),
    ("Cat III (2025)", 11100, 12250, "#d62728"),
    ("Cat III (2026)", 13050, 13550, "#9467bd"),
]

print("Environment initialized successfully.")


---
## 1. Data Ingestion & Goodness-of-Fit Quality Selection

We ingest all 73 runs from `/eos/user/i/idioniso/Timing/cscint/fits/run*_cscint.json`.
To eliminate unconstrained slopes and low-statistics artifacts, we apply standard **Goodness-of-Fit quality requirements**:
- **Upstream (US):** $12.0 \le c_{\mathrm{scint}} \le 16.0\,\mathrm{cm/ns}$, $\delta c_{\mathrm{fit}} \le 0.10\,\mathrm{cm/ns}$, $\delta c_{\mathrm{tot}} \le 0.60\,\mathrm{cm/ns}$, $\chi^2/\mathrm{ndf} \le 15.0$.
- **Downstream (DS):** $13.0 \le c_{\mathrm{scint}} \le 16.5\,\mathrm{cm/ns}$, $\delta c_{\mathrm{fit}} \le 0.10\,\mathrm{cm/ns}$, $\delta c_{\mathrm{tot}} \le 0.80\,\mathrm{cm/ns}$, $\chi^2/\mathrm{ndf} \le 20.0$.
- **Veto:** $11.0 \le c_{\mathrm{scint}} \le 18.5\,\mathrm{cm/ns}$, $\delta c_{\mathrm{fit}} \le 0.15\,\mathrm{cm/ns}$, $\delta c_{\mathrm{tot}} \le 1.20\,\mathrm{cm/ns}$.


In [ ]:
fits_dir = "/eos/user/i/idioniso/Timing/cscint/fits"
json_files = sorted(glob.glob(os.path.join(fits_dir, "run*_cscint.json")))

def get_run_period(run: int) -> str:
    if 5000 <= run < 6000:
        return "Cat II (2022)"
    elif 6000 <= run < 7000:
        return "Cat III (2023)"
    elif 8000 <= run < 10000:
        return "Cat III (2024)"
    elif 10000 <= run < 13000:
        return "Cat III (2025)"
    elif 13000 <= run < 14000:
        return "Cat III (2026)"
    return "Other"

def parse_file(jf):
    r = int(os.path.basename(jf).replace("run", "").replace("_cscint.json", ""))
    period = get_run_period(r)
    with open(jf) as fp:
        d = json.load(fp)
    
    recs = []
    # US & DS from DS reference
    for ch_key, info in d.get("DS", {}).items():
        det_id = info.get("detID", 0)
        subsys = det_id // 10000
        plane = (det_id % 10000) // 1000
        bar = det_id % 1000
        sipm = info.get("sipm", 0)
        c = info.get("c_scint", np.nan)
        err = info.get("delta_c_tot", np.nan)
        fit_err = info.get("delta_c_fit", np.nan)
        syst_err = info.get("delta_c_syst", np.nan)
        chi2ndf = info.get("chi2_ndf", np.nan)
        s_name = "US" if subsys == 2 else ("DS" if subsys == 3 else "Other")
        if s_name in ("US", "DS"):
            side = "Left" if (sipm < 8 if s_name == "US" else sipm == 0) else "Right"
            recs.append({
                "run": r, "period": period, "channel": ch_key, "detID": det_id,
                "subsys": s_name, "plane": plane, "bar": bar, "sipm": sipm,
                "side": side, "c_scint": c, "delta_c_tot": err,
                "delta_c_fit": fit_err, "delta_c_syst": syst_err, "chi2ndf": chi2ndf
            })
            
    # Veto from SF reference
    for ch_key, info in d.get("SF", {}).items():
        det_id = info.get("detID", 0)
        subsys = det_id // 10000
        plane = (det_id % 10000) // 1000
        bar = det_id % 1000
        sipm = info.get("sipm", 0)
        c = info.get("c_scint", np.nan)
        err = info.get("delta_c_tot", np.nan)
        fit_err = info.get("delta_c_fit", np.nan)
        syst_err = info.get("delta_c_syst", np.nan)
        chi2ndf = info.get("chi2_ndf", np.nan)
        if subsys == 1:
            side = "Left" if sipm < 8 else "Right"
            recs.append({
                "run": r, "period": period, "channel": ch_key, "detID": det_id,
                "subsys": "Veto", "plane": plane, "bar": bar, "sipm": sipm,
                "side": side, "c_scint": c, "delta_c_tot": err,
                "delta_c_fit": fit_err, "delta_c_syst": syst_err, "chi2ndf": chi2ndf
            })
    return recs

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    res = list(ex.map(parse_file, json_files))

df_all = pd.DataFrame([row for r in res for row in r])

# Apply Goodness-of-Fit requirements per subsystem
cond_us = (
    (df_all["subsys"] == "US") &
    (df_all["c_scint"].between(12.0, 16.0)) &
    (df_all["delta_c_fit"] <= 0.10) &
    (df_all["delta_c_tot"] <= 0.80)
)
cond_ds = (
    (df_all["subsys"] == "DS") &
    (df_all["c_scint"].between(13.0, 16.5)) &
    (df_all["delta_c_fit"] <= 0.10) &
    (df_all["delta_c_tot"] <= 0.85)
)
cond_veto = (
    (df_all["subsys"] == "Veto") &
    (df_all["c_scint"].between(11.0, 18.5)) &
    (df_all["delta_c_fit"] <= 0.15) &
    (df_all["delta_c_tot"] <= 1.25)
)
df_good = df_all[cond_us | cond_ds | cond_veto].copy()

print(f"Ingested {len(df_all):,} raw channel records.")
print(f"Retained {len(df_good):,} good-quality converged fits ({len(df_good)/len(df_all)*100:.1f}%) across {df_good['run'].nunique()} runs.")
print("\nGood fits breakdown by subsystem:")
for sub in ["US", "DS", "Veto"]:
    sub_raw = df_all[df_all["subsys"] == sub]
    sub_g = df_good[df_good["subsys"] == sub]
    print(f"  - {sub:5s}: {len(sub_g):5,} good fits / {len(sub_raw):5,} raw ({len(sub_g)/len(sub_raw)*100:4.1f}%) | "
          f"Raw Std: {sub_raw['c_scint'].std():.3f} -> Good Std: {sub_g['c_scint'].std():.3f} cm/ns")


---
## 2. Intra-Bar SiPM Variance Across Subsystems & Operational Years

For every physical bar in every run, we compute the standard deviation $\sigma_{\mathrm{bar}}$ across all SiPMs attached to that bar:
$$\sigma_{\mathrm{bar}} = \sqrt{\frac{1}{K - 1} \sum_{k=1}^K (c_k - \bar{c}_{\mathrm{bar}})^2}$$
We plot $\sigma_{\mathrm{bar}}$ for each system (**US**, **DS**, and **Veto**), superimposing the **bar number as a colormap** on the run timeline, and compare intra-bar dispersion across each operational year.


In [ ]:
# Compute per-bar variance metrics across all SiPMs of each bar in each run
bar_records = []
for (run, period, subsys, det_id, bar), group in df_good.groupby(["run", "period", "subsys", "detID", "bar"]):
    if len(group) < 2:
        continue
    vals = group["c_scint"].values
    bar_records.append({
        "run": run, "period": period, "subsys": subsys, "detID": det_id, "bar": bar,
        "n_sipms": len(group),
        "bar_mean": float(np.mean(vals)),
        "bar_std": float(np.std(vals, ddof=1)),
        "bar_var": float(np.var(vals, ddof=1)),
        "bar_iqr": float(np.percentile(vals, 75) - np.percentile(vals, 25))
    })

df_bar_var = pd.DataFrame(bar_records)
print(f"Evaluated intra-bar SiPM variance for {len(df_bar_var):,} bar-run observations.")

# Multi-panel figure 1: 3 vertically stacked plots (US, DS, Veto) with Bar Colormap
fig, (ax_us, ax_ds, ax_veto) = plt.subplots(3, 1, figsize=(15, 12), sharex=True, layout="constrained")

plot_cfgs = [
    (ax_us,   "US",   10, (0.0, 1.4), r"Upstream (US) Bars: Intra-Bar $\sigma(c_{\mathrm{scint}})$ Across SiPMs"),
    (ax_ds,   "DS",   60, (0.0, 1.8), r"Downstream (DS) Bars: Intra-Bar $\sigma(c_{\mathrm{scint}})$ Across SiPMs"),
    (ax_veto, "Veto",  7, (0.0, 2.5), r"Veto Bars: Intra-Bar $\sigma(c_{\mathrm{scint}})$ Across SiPMs"),
]

for ax, subsys_name, n_bars, ylim, title in plot_cfgs:
    for p_name, x0, x1, col in period_spans:
        ax.axvspan(x0, x1, color=col, alpha=0.08, zorder=1)
        
    sub_bars = df_bar_var[df_bar_var["subsys"] == subsys_name]
    cmap = plt.colormaps["turbo"].resampled(n_bars)
    norm = mcolors.Normalize(vmin=-0.5, vmax=n_bars - 0.5)
    
    sc = ax.scatter(
        sub_bars["run"], sub_bars["bar_std"],
        c=sub_bars["bar"], cmap=cmap, norm=norm,
        alpha=0.65, s=24, edgecolors="none", rasterized=True, zorder=3
    )
    
    run_med = sub_bars.groupby("run")["bar_std"].median()
    ax.plot(
        run_med.index, run_med.values, color="black", marker="D", markersize=3.5,
        linestyle="-", linewidth=1.2, alpha=0.9, label=r"Run Median $\sigma_{\mathrm{bar}}$", zorder=4
    )
    
    glob_med = float(np.median(sub_bars["bar_std"].dropna()))
    ax.axhline(glob_med, color="dimgray", linestyle="--", linewidth=1.2,
               label=fr"Global Median Intra-Bar $\sigma$: {glob_med:.3f} cm/ns", zorder=2)
    
    ax.set_ylabel(r"Intra-Bar $\sigma(c_{\mathrm{scint}})$ [cm/ns]", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=5)
    ax.set_ylim(ylim)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="upper right", framealpha=0.92, fontsize=9.5)
    
    cbar = fig.colorbar(sc, ax=ax, orientation="vertical", fraction=0.02, pad=0.015)
    cbar.set_label(f"{subsys_name} Bar Index", fontsize=10)

ax_veto.set_xlabel("Run Number", fontsize=12)
fig.suptitle(r"Intra-Bar SiPM $c_{\mathrm{scint}}$ Variance Across Subsystems with Superimposed Bar Colormap", fontsize=14, fontweight="bold", y=1.008)
plt.show()

# Multi-panel figure 2: Boxplot comparing intra-bar variance by year for each subsystem
fig, (ax_b_us, ax_b_ds, ax_b_veto) = plt.subplots(1, 3, figsize=(16, 5))
era_box_cfgs = [
    (ax_b_us,   "US",   r"US Intra-Bar $\sigma_{\mathrm{bar}}$ by Year",   (0.0, 1.2)),
    (ax_b_ds,   "DS",   r"DS Intra-Bar $\sigma_{\mathrm{bar}}$ by Year",   (0.0, 1.4)),
    (ax_b_veto, "Veto", r"Veto Intra-Bar $\sigma_{\mathrm{bar}}$ by Year", (0.0, 1.8)),
]

for ax_b, s_name, b_title, y_lim in era_box_cfgs:
    s_bars = df_bar_var[df_bar_var["subsys"] == s_name]
    box_data = [s_bars[s_bars["period"] == p]["bar_std"].dropna().values for p in periods_ordered]
    bp = ax_b.boxplot(
        box_data, labels=[period_labels_short[p] for p in periods_ordered],
        patch_artist=True, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5}
    )
    for patch, p in zip(bp["boxes"], periods_ordered):
        patch.set_facecolor(period_colors[p])
        patch.set_alpha(0.65)
    ax_b.set_ylabel(r"Intra-Bar $\sigma_{\mathrm{bar}}$ [cm/ns]", fontsize=11)
    ax_b.set_title(b_title, fontsize=12, fontweight="bold", pad=6)
    ax_b.set_ylim(y_lim)
    ax_b.grid(True, linestyle=":", alpha=0.5)

fig.suptitle(r"Year-by-Year Intra-Bar SiPM Dispersion Across Operational Eras", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


---
## 3. Upstream (US) Bars: Variance Comparison (Left vs. Right vs. Left+Right) Per Year

We now analyze the variance of $c_{\mathrm{scint}}$ across all Upstream (US) bars stratified by year under three readout configurations:
1. **Left SiPMs Only (0–7)**
2. **Right SiPMs Only (8–15)**
3. **Combined (Left + Right)**


In [ ]:
df_us_good = df_good[df_good["subsys"] == "US"].copy()

# Summary Table: US Left vs Right vs All by Year
us_era_rows = []
for p in periods_ordered:
    pg = df_us_good[df_us_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    us_era_rows.append({
        "Year / Period": p,
        "Left Mean ± Std": f"{np.mean(c_l):.2f} ± {np.std(c_l, ddof=1):.3f} cm/ns (N={len(c_l):,})",
        "Right Mean ± Std": f"{np.mean(c_r):.2f} ± {np.std(c_r, ddof=1):.3f} cm/ns (N={len(c_r):,})",
        "All Mean ± Std": f"{np.mean(c_all):.2f} ± {np.std(c_all, ddof=1):.3f} cm/ns (N={len(c_all):,})",
        "Left-Right Delta": f"{np.mean(c_l) - np.mean(c_r):+.3f} cm/ns",
        "CV [%]": f"{np.std(c_all, ddof=1)/np.mean(c_all)*100:.2f}%"
    })

print("=== Upstream (US) Readout Variance by Operational Year ===")
print(pd.DataFrame(us_era_rows).to_string(index=False))

# Visual Comparison: US Left vs Right vs All Across Eras
fig, axes = plt.subplots(1, 5, figsize=(18, 4.5), sharey=True, layout="constrained")

for ax, p in zip(axes, periods_ordered):
    pg = df_us_good[df_us_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    bp = ax.boxplot(
        [c_l, c_r, c_all],
        labels=["Left", "Right", "All"],
        patch_artist=True, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5}
    )
    bp["boxes"][0].set_facecolor("#1f77b4")
    bp["boxes"][1].set_facecolor("#ff7f0e")
    bp["boxes"][2].set_facecolor("#2ca02c")
    for box in bp["boxes"]: box.set_alpha(0.65)
    
    ax.set_title(p, fontsize=11, fontweight="bold", pad=5)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_ylim(12.2, 15.8)

axes[0].set_ylabel(r"$c_{\mathrm{scint}}$ [cm/ns]", fontsize=11)
fig.suptitle(r"Upstream (US) Propagation Speed by Year: Left vs. Right vs. Combined Readout", fontsize=13, fontweight="bold", y=1.02)
plt.show()


---
## 4. Veto Bars: Variance Comparison (Left vs. Right vs. Left+Right) Per Year

Evaluates the variance of extracted $c_{\mathrm{scint}}$ across all Veto bars stratified by year under:
1. **Left SiPMs Only (0–7)**
2. **Right SiPMs Only (8–15)**
3. **Combined (Left + Right)**


In [ ]:
df_veto_good = df_good[df_good["subsys"] == "Veto"].copy()

# Summary Table: Veto Left vs Right vs All by Year
veto_era_rows = []
for p in periods_ordered:
    pg = df_veto_good[df_veto_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    veto_era_rows.append({
        "Year / Period": p,
        "Left Mean ± Std": f"{np.mean(c_l):.2f} ± {np.std(c_l, ddof=1):.3f} cm/ns (N={len(c_l):,})",
        "Right Mean ± Std": f"{np.mean(c_r):.2f} ± {np.std(c_r, ddof=1):.3f} cm/ns (N={len(c_r):,})",
        "All Mean ± Std": f"{np.mean(c_all):.2f} ± {np.std(c_all, ddof=1):.3f} cm/ns (N={len(c_all):,})",
        "Left-Right Delta": f"{np.mean(c_l) - np.mean(c_r):+.3f} cm/ns",
        "CV [%]": f"{np.std(c_all, ddof=1)/np.mean(c_all)*100:.2f}%"
    })

print("=== Veto Readout Variance by Operational Year ===")
print(pd.DataFrame(veto_era_rows).to_string(index=False))

# Visual Comparison: Veto Left vs Right vs All Across Eras
fig, axes = plt.subplots(1, 5, figsize=(18, 4.5), sharey=True, layout="constrained")

for ax, p in zip(axes, periods_ordered):
    pg = df_veto_good[df_veto_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    bp = ax.boxplot(
        [c_l, c_r, c_all],
        labels=["Left", "Right", "All"],
        patch_artist=True, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5}
    )
    bp["boxes"][0].set_facecolor("#1f77b4")
    bp["boxes"][1].set_facecolor("#ff7f0e")
    bp["boxes"][2].set_facecolor("#2ca02c")
    for box in bp["boxes"]: box.set_alpha(0.65)
    
    ax.set_title(p, fontsize=11, fontweight="bold", pad=5)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_ylim(11.0, 18.0)

axes[0].set_ylabel(r"$c_{\mathrm{scint}}$ [cm/ns]", fontsize=11)
fig.suptitle(r"Veto Propagation Speed by Year: Left vs. Right vs. Combined Readout", fontsize=13, fontweight="bold", y=1.02)
plt.show()


---
## 5. Downstream (DS) Bars: Variance Comparison (Left vs. Right vs. Left+Right) Per Year

Evaluates the variance of extracted $c_{\mathrm{scint}}$ across all Downstream (DS) bars stratified by year under:
1. **Left SiPMs Only (Ch 0)**
2. **Right SiPMs Only (Ch 1)**
3. **Combined (Left + Right)**


In [ ]:
df_ds_good = df_good[df_good["subsys"] == "DS"].copy()

# Summary Table: DS Left vs Right vs All by Year
ds_era_rows = []
for p in periods_ordered:
    pg = df_ds_good[df_ds_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    ds_era_rows.append({
        "Year / Period": p,
        "Left Mean ± Std": f"{np.mean(c_l):.2f} ± {np.std(c_l, ddof=1):.3f} cm/ns (N={len(c_l):,})",
        "Right Mean ± Std": f"{np.mean(c_r):.2f} ± {np.std(c_r, ddof=1):.3f} cm/ns (N={len(c_r):,})",
        "All Mean ± Std": f"{np.mean(c_all):.2f} ± {np.std(c_all, ddof=1):.3f} cm/ns (N={len(c_all):,})",
        "Left-Right Delta": f"{np.mean(c_l) - np.mean(c_r):+.3f} cm/ns",
        "CV [%]": f"{np.std(c_all, ddof=1)/np.mean(c_all)*100:.2f}%"
    })

print("=== Downstream (DS) Readout Variance by Operational Year ===")
print(pd.DataFrame(ds_era_rows).to_string(index=False))

# Visual Comparison: DS Left vs Right vs All Across Eras
fig, axes = plt.subplots(1, 5, figsize=(18, 4.5), sharey=True, layout="constrained")

for ax, p in zip(axes, periods_ordered):
    pg = df_ds_good[df_ds_good["period"] == p]
    c_l = pg[pg["side"] == "Left"]["c_scint"].dropna()
    c_r = pg[pg["side"] == "Right"]["c_scint"].dropna()
    c_all = pg["c_scint"].dropna()
    
    bp = ax.boxplot(
        [c_l, c_r, c_all],
        labels=["Left", "Right", "All"],
        patch_artist=True, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5}
    )
    bp["boxes"][0].set_facecolor("#1f77b4")
    bp["boxes"][1].set_facecolor("#ff7f0e")
    bp["boxes"][2].set_facecolor("#2ca02c")
    for box in bp["boxes"]: box.set_alpha(0.65)
    
    ax.set_title(p, fontsize=11, fontweight="bold", pad=5)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_ylim(13.0, 16.5)

axes[0].set_ylabel(r"$c_{\mathrm{scint}}$ [cm/ns]", fontsize=11)
fig.suptitle(r"Downstream (DS) Propagation Speed by Year: Left vs. Right vs. Combined Readout", fontsize=13, fontweight="bold", y=1.02)
plt.show()


---
## 6. Master Cross-Category Synthesis Table

Consolidates the effective speed, intra-bar variance, and relative dispersion for all 5 operational eras across all subsystems.


In [ ]:
master_rows = []
for p in periods_ordered:
    for sub in ["US", "DS", "Veto"]:
        sub_p = df_good[(df_good["period"] == p) & (df_good["subsys"] == sub)]
        if len(sub_p) == 0: continue
        c_l = sub_p[sub_p["side"] == "Left"]["c_scint"].dropna()
        c_r = sub_p[sub_p["side"] == "Right"]["c_scint"].dropna()
        c_all = sub_p["c_scint"].dropna()
        bar_stds = df_bar_var[(df_bar_var["period"] == p) & (df_bar_var["subsys"] == sub)]["bar_std"].dropna()
        
        master_rows.append({
            "Year / Period": p,
            "Subsystem": sub,
            "Left Mean ± Std": f"{np.mean(c_l):.2f} ± {np.std(c_l, ddof=1):.3f}",
            "Right Mean ± Std": f"{np.mean(c_r):.2f} ± {np.std(c_r, ddof=1):.3f}",
            "Combined Mean ± Std": f"{np.mean(c_all):.2f} ± {np.std(c_all, ddof=1):.3f}",
            "Intra-Bar Median Std": f"{np.median(bar_stds):.3f} cm/ns",
            "CV [%]": f"{np.std(c_all, ddof=1)/np.mean(c_all)*100:.2f}%"
        })

df_master_synth = pd.DataFrame(master_rows)
print("=== Master Multi-Subsystem Cross-Category Synthesis Table ===")
print(df_master_synth.to_string(index=False))
